In [ ]:
# ============================================================
# Cell 0: 检查 GPU 是否可用
# ============================================================
# 大模型示例最常见的失败原因不是代码错，而是没切到 GPU 运行时。
# 先一眼确认 GPU 在线，再继续后面的步骤，避免白白下载几 GB 权重。
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
    # 显存以 GB 为单位打印，T4 应显示约 14.7 GB
    total_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"Total VRAM: {total_mem:.1f} GB")
else:
    # 切换方式：菜单 → 代码执行程序 → 更改运行时类型 → 硬件加速器 → T4 GPU
    print("⚠️ 当前没有 GPU，请切到 GPU 运行时再继续")

In [ ]:
%%capture
# ============================================================
# Cell 1: 安装/升级依赖库
# ============================================================
# 重要：%%capture 是单元格魔法命令，必须是 cell 的第一行（前面不能有任何内容，包括注释）
# 否则 Jupyter / VS Code 会将其当作 line magic 解析，报错：
#   UsageError: Line magic function `%%capture` not found.
# 它的作用：捕获整个 cell 的输出，让 pip install 几十行的安装日志不显示在屏幕上
# ! 前缀让 IPython 把这一行交给系统 shell 执行（而非 Python 解释器）
# -q 表示 quiet 模式，进一步减少 pip 的输出噪音
# -U 表示 upgrade，如果已装则升级到最新版
# transformers>=4.51:  Qwen3 系列要求至少 4.51，否则 chat template 不识别 enable_thinking
# accelerate:           分布式/混合精度加速，加载大模型时几乎是必需依赖
# bitsandbytes:         提供 8-bit / 4-bit 量化能力，让大模型在小显存上跑起来
!pip install -q -U "transformers>=4.51" accelerate bitsandbytes

In [ ]:
# ============================================================
# Cell 2: 加载模型与分词器（采用 4-bit 量化）
# ============================================================
# AutoModelForCausalLM: "自动选择 CausalLM 类"的工厂类
#   CausalLM = Causal Language Model，因果语言模型，即 GPT 这类自回归生成模型
# AutoTokenizer:        根据模型 ID 自动选择对应的分词器实现
# BitsAndBytesConfig:   配置 bitsandbytes 的量化参数
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

# Hugging Face Hub 模型 ID，格式 "组织名/模型名"
# Qwen3 系列把 base 与 chat 合并到同一仓库，仓库名不再带 "-Instruct" 后缀：
#   "Qwen/Qwen3-8B"      → chat 模型（本教程使用）
#   "Qwen/Qwen3-8B-Base" → 基座，只做续写不按对话回答
model_id = "Qwen/Qwen3-8B"

# 4-bit 量化配置（原理与手算示例见「关于 4-bit 量化」一节）
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,                       # 启用 4-bit 量化
    bnb_4bit_compute_dtype=torch.float16,    # 反量化后矩阵乘法的 dtype（T4 不原生支持 BF16；L4/A100 可改 torch.bfloat16）
    bnb_4bit_quant_type="nf4",               # 量化点采用 NF4（NormalFloat4）
    bnb_4bit_use_double_quant=True,          # 启用双量化，再把每块的 scale 也量化一次
)

# from_pretrained() 自动从 Hugging Face Hub 下载权重并缓存到 ~/.cache/huggingface/
# 下次加载同一模型不会重复下载
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",                # 让 accelerate 自动分配模型层；单卡时等价于全部放到 GPU
    quantization_config=quant_config, # 应用上面定义的 4-bit 量化配置
)

# 加载与模型配对的分词器：负责文本 ↔ token id 的双向转换
tokenizer = AutoTokenizer.from_pretrained(model_id)

In [ ]:
# ============================================================
# Cell 3: 构造对话并让模型生成回答（非思考模式 / 普通对话）
# ============================================================
# Qwen3 chat 模型期望的输入是"对话格式"，每条消息有 role 和 content
# role 通常是 "system"（系统提示）/ "user"（用户）/ "assistant"（模型回答）
messages = [
    {"role": "system", "content": "你是一个乐于助人的 AI 助手，回答简洁明了。"},
    {"role": "user", "content": "用一句话解释什么是 Transformer 架构。"},
]

# apply_chat_template() 会按照模型训练时使用的对话模板格式化输入
# 不同模型的模板不同（Qwen / Llama / Phi 各自有自己的特殊 token）
# add_generation_prompt=True 在末尾追加"轮到 assistant 回答"的提示符
# enable_thinking=False 关闭"思考模式"——直接输出答案，不先打 <think>...</think> 推理过程
#   想要更强的逻辑/数学/代码能力时，把它改成 True，并提高 max_new_tokens 给足思考空间
# return_tensors="pt" 直接返回 PyTorch 张量（pt = PyTorch）
# return_dict=True 让返回类型固定为 BatchEncoding（含 input_ids、attention_mask）
#   注意：新版 transformers (≥4.45) 在某些情况下默认返回 BatchEncoding 而非 Tensor
#   若把 BatchEncoding 直接作为位置参数传给 generate()，会触发：
#     KeyError: 'shape' → AttributeError
#   所以推荐显式 return_dict=True + 用 **inputs 解包
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    enable_thinking=False,
    return_tensors="pt",
    return_dict=True,
).to(model.device)  # 把输入张量移到模型所在设备，与模型在同一设备才能计算

# Qwen 系列对话终止符是 <|im_end|>，与一般模型的 EOS 不同。
# 多数 transformers 版本里 tokenizer.eos_token_id 已经被 chat template 改写成 <|im_end|>，
# 但严格起见显式查一下 token id 更稳，避免某些版本下模型不停止：
im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")

# generate() 是 transformers 的统一文本生成接口
# torch.no_grad() 表示推理过程不计算梯度，节省显存并加速
with torch.no_grad():
    outputs = model.generate(
        # **inputs 把字典解包为 input_ids=..., attention_mask=...
        # 显式传 attention_mask 还能避免 padding 干扰，比只传 input_ids 更稳
        **inputs,
        # max_new_tokens 限制最多生成多少个新 token（不含输入部分）
        # 非思考模式 200 足够；思考模式因为要先输出推理过程，建议 ≥ 2048
        max_new_tokens=200,
        # Qwen3 官方推荐：非思考模式用 temperature=0.7、top_p=0.8、top_k=20
        # ⚠️ 思考模式（enable_thinking=True）下不能用贪心解码（do_sample=False），
        #    会出现重复退化；必须 do_sample=True 配 temperature=0.6、top_p=0.95
        do_sample=True,
        temperature=0.7,
        top_p=0.8,
        top_k=20,
        # eos_token_id 标记生成结束的 token，遇到它就停止
        # 同时传 tokenizer.eos_token_id 和 <|im_end|> 双保险
        eos_token_id=[tokenizer.eos_token_id, im_end_id],
    )

# outputs 形状是 [batch_size, sequence_length]，包含输入 + 生成的全部 token
# 我们只想看新生成的部分，所以从输入长度处切片
input_length = inputs["input_ids"].shape[-1]
generated_tokens = outputs[0][input_length:]

# decode() 把 token id 列表还原成可读字符串
# skip_special_tokens=True 去掉 <|im_end|> 等特殊标记
response = tokenizer.decode(generated_tokens, skip_special_tokens=True)

print(response)